# EpstLoop on a Colab T4

Development stays in VS Code; this notebook only **runs** the GPU stage and sends results back.

1. *Runtime → Change runtime type → T4 GPU*, then run the cells top to bottom.
2. Each run writes into `results/`. The last cell zips them for download. Unzip into the local repo's `results/` folder, and the analysis (`calibration_study.py`) runs locally on CPU.

If a cell fails, copy the full error back to Claude in VS Code.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Get the code (re-run after every push from VS Code)

In [2]:
import os
REPO = "https://github.com/chiruuu21/epistemic-architecture.git"
if not os.path.exists("/content/epistemic-architecture"):
    !git clone -q {REPO} /content/epistemic-architecture
%cd /content/epistemic-architecture
!git pull -q && git log --oneline -1

/content/epistemic-architecture
94fd0bd (HEAD -> main, origin/main, origin/HEAD) Reworded TriviaQA pipeline; drop duplicate questions from the calibration study


## 2. Install and cache models on Google Drive
Caching on Drive means a disconnected session does not re-download the models. Skip the Drive part if you prefer; everything still works without it.

In [3]:
!pip install -q -r requirements-colab.txt
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| cuda", torch.cuda.is_available())

Mounted at /content/drive
torch 2.14.1+cu130 | transformers 5.17.0 | cuda True


## 3. Sanity check: the offline test suite (should say ALL 87 CHECKS PASSED)

In [4]:
!python run_all_tests.py

                             EpstLoop test suite                              

  test_belief_store.py         PASS   5 checks    0.35s
    Belief store: weighting, state, RIF floor, batched query

  test_reconsolidation.py      PASS   7 checks    0.33s
    Reconsolidation: contest/reinforce branches, RIF, skips

  test_nli_backend.py          PASS  16 checks    0.32s
    NLI backend: label maps, thresholds, abstention, batching

  test_ept_loop.py             PASS  18 checks    6.41s
    End-to-end loop: claim extraction, flag accumulation

  test_sae_loader.py           PASS   8 checks    3.57s
    SAE loader: tensor mapping, b_dec, TopK, refusals

  test_context_retrieval.py    PASS   8 checks    4.89s
    Context retrieval: running mean, relevance cutoff, warm-up

  test_calibration.py          PASS   8 checks    3.69s
    Calibration: ECE/AUROC/Platt, TriviaQA matching, graded PIK

  test_convergence_logic.py    PASS  13 checks    2.72s
    Trust retrieval, reconsolidation revisio

In [ ]:
from google.colab import userdata
import os

# Retrieve token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Make token available to Hugging Face libraries
os.environ["HF_TOKEN"] = hf_token

# Authenticate
from huggingface_hub import login, whoami

login(token=hf_token)

# Verify authentication
print(whoami())

## 4. GPU stage: GPT-2 small (the project's model), 3,000 TriviaQA questions
Takes a few minutes. The final printout includes `accuracy_em` and the loop check, which should be about 1e-3 or smaller.

In [ ]:
!python collect_triviaqa.py --model openai-community/gpt2 --n 3000

Resolving data files: 100% 26/26 [00:00<00:00, 20254.81it/s]
3000 questions; loading openai-community/gpt2 (float16) on cuda:0
Loading weights: 100% 148/148 [00:00<00:00, 187.00it/s]
layer 8, d_model 768
answers done (54s)
keys done (65s)
loop check: max |loop cos - pooled cos| over 32 questions = 1.19e-04
{
  "model": "openai-community/gpt2",
  "layer": 8,
  "d_model": 768,
  "dtype": "float16",
  "device": "cuda:0",
  "n": 3000,
  "seed": 0,
  "accuracy_em": 0.048666666666666664,
  "loop_check": {
    "n": 32,
    "max_abs_diff": 0.00011914968490600586
  },
  "seconds": 69.8,
  "torch": "2.14.1+cu130",
  "transformers": "5.17.0",
  "gpu": "Tesla T4"
}


## 5. GPU stage: Qwen2.5-3B, the same 3,000 questions
A model that can actually answer trivia, so that ECE has populated bins. It runs in fp16 (about 6 GB). If it stops with *non-finite hidden states*, run the float32 cell below it instead (about 12 GB, still fits on a T4).

In [ ]:
!python collect_triviaqa.py --model Qwen/Qwen2.5-3B --n 3000 --bs 16

Resolving data files: 100% 26/26 [00:00<00:00, 6912.08it/s]
3000 questions; loading Qwen/Qwen2.5-3B (float16) on cuda:0
config.json: 100% 683/683 [00:00<00:00, 3.02MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 15.1MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 11.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 13.8MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 21.9MB/s]
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 72.5MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/2.20G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  29% 1.80G/6.17G [00:15<00:37, 115MB/s,  135MB/s  ]

Reconstructing (incomplete total...):  88% 5.44G/6.17G [00:42<00:07, 97.0MB/s,  189MB/s  ]
Reconstructing (incomplete total...): 100% 6.17G/6.17G [00:45<00:00, 132MB/s, 97.0MB/s  ] 

Fetching 2 files: 100% 2/2 [00:47<00:00, 23.53s/it]
Dow

In [ ]:
# only if the fp16 run failed with "non-finite hidden states"
# !python collect_triviaqa.py --model Qwen/Qwen2.5-3B --n 3000 --dtype float32 --bs 8

## 6. Analysis (CPU; also runs locally from the same files)

In [ ]:
!python calibration_study.py --tag gpt2
!python calibration_study.py --tag qwen2.5-3b

Calibration study: openai-community/gpt2 (layer 8, float16, Tesla T4)
TriviaQA rc.nocontext validation, n=3000 (seed 0); calibration/test 50/50; 95% bootstrap CIs over test questions
loop check: |EpTLoop last-token cosine - pooled cosine| <= 1.2e-04 over 32 questions

A. LM alone (closed-book, 5-shot, greedy). Accuracy (EM) 0.049; lenient match 0.058
  confidence                           acc   conf             ECE [95% CI]  Brier   AUROC right>wrong [CI]
  answer probability (raw)           0.045  0.058     0.023 [0.015, 0.034]  0.043     0.770 [0.721, 0.818]
  answer probability, Platt-scaled   0.045  0.053     0.011 [0.004, 0.021]  0.041     0.770 [0.721, 0.818]
  constant = calib accuracy          0.045  0.052     0.007 [0.001, 0.017]  0.043     0.500 [0.500, 0.500]
  The constant row is the sanity check: near-zero ECE with AUROC 0.5. A low ECE
  only counts if AUROC is clearly above 0.5 as well.

B. The loop on TriviaQA (1500 beliefs stored, half the questions)
  relevance AUROC, 

In [6]:
import shutil
shutil.rmtree("/content/drive/MyDrive/hf_cache/hub/models--Qwen--Qwen2.5-3B-Instruct", ignore_errors=True)
!df -h /content/drive/MyDrive | tail -1


drive           113G   12G  102G  11% /content/drive


## 7. Hard retrieval: reworded questions
The first study looked each belief up with the same wording it was stored under. Here an instruction-tuned model rewrites every question (a paraphrase, and the fact as a statement), and both models are tested on the reworded versions.

**Sections 4-6 do not need re-running** for this: their output files are already in the repo. Run sections 1-3, then the three cells below, then section 8.

Step 1 (about 15 minutes): reword the questions with Qwen2.5-3B-Instruct. It prints eight examples at the end; `both_ok` should be above about 0.8.

In [7]:
!HF_HOME=/content/hf_local python reword_triviaqa.py


2780 questions from results/triviaqa_gpt2.jsonl; loading Qwen/Qwen2.5-3B-Instruct (float16) on cuda:0
config.json: 100% 661/661 [00:00<00:00, 3.30MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 25.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 65.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 110MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 122MB/s]
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 70.3MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.97G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   1% 92.1M/6.17G [00:01<01:37, 62.7MB/s, 53.6MB/s  ]
Reconstructing (incomplete total...):  25% 1.53G/6.17G [00:16<01:00, 77.0MB/s,  138MB/s  ]

Reconstructing (incomplete total...):  88% 5.40G/6.17G [00:40<00:05, 139MB/s,  216MB/s  ]
Recons

Step 2 (a few minutes each): keys and answers for the reworded text, same layer as before. `accuracy_em_paraphrase` should be close to `accuracy_em_original`; a big drop means the paraphrases changed the questions' meaning.

In [9]:
!HF_HOME=/content/hf_local python collect_triviaqa.py --model Qwen/Qwen2.5-3B --reworded --bs 16


2780 reworded questions; loading Qwen/Qwen2.5-3B (float16) on cuda:0
config.json: 100% 683/683 [00:00<00:00, 3.17MB/s]
tokenizer_config.json: 100% 7.23k/7.23k [00:00<00:00, 22.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 70.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 96.8MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 128MB/s]
model.safetensors.index.json: 100% 35.6k/35.6k [00:00<00:00, 86.6MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.17G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  16% 967M/6.17G [00:06<00:28, 185MB/s,  212MB/s  ]
Reconstructing (incomplete total...):  19% 1.16G/6.17G [00:07<00:30, 162MB/s,  131MB/s  ]
Reconstructing (incomplete total...):  25% 1.54G/6.17G [00:14<01:30, 51.4MB/s,  101MB/s  ]
Reconstructing (incomplete total...):  73% 4.53G/6.17G [00:37<00:11, 140MB/s,  136MB/s  ]

Reconstructing (inco

Step 3 (CPU): the comparison, easy wording against hard wording.

In [10]:
!python reworded_study.py --tag gpt2
!python reworded_study.py --tag qwen2.5-3b

Reworded TriviaQA study: openai-community/gpt2 (layer 8)
rewordings by Qwen/Qwen2.5-3B-Instruct; 2597 of 2780 questions passed both checks (93.4%)
content-word overlap, original vs paraphrased question: mean 0.57 (quartiles 0.43 / 0.57 / 0.71)
meaning check: model accuracy on the original wording 0.049, on the paraphrase 0.051

A. Retrieval and the FOK decision (test half)
  condition                   top-1  own cos  rel AUROC   theta  FOK stored  FOK unstored
  1 same Q -> Q+A belief      0.997    0.886      0.993   0.552       0.006         0.731
  2 new Q  -> Q+A belief      0.874    0.712      0.886   0.374       0.006         0.107
  3 same Q -> statement       0.967    0.708      0.931   0.386       0.020         0.363
  4 new Q  -> statement       0.911    0.638      0.862   0.352       0.025         0.224
  top-1: the stored question retrieves its own belief. own cos: its mean match cosine.
  rel AUROC: best-match cosine separates stored from unstored questions.
  FOK stored /

## 8. Bring the results back to VS Code

In [11]:
import glob, shutil, zipfile
files = sorted(glob.glob("results/triviaqa_*") + glob.glob("results/calibration_*") + glob.glob("results/reworded_*"))
with zipfile.ZipFile("epstloop_results.zip", "w") as z:
    for f in files:
        z.write(f)
print("\n".join(files))
os.makedirs("/content/drive/MyDrive/EpstLoop_results", exist_ok=True)
shutil.copy("epstloop_results.zip", "/content/drive/MyDrive/EpstLoop_results/")
from google.colab import files as colab_files
colab_files.download("epstloop_results.zip")

results/calibration_gpt2.json
results/calibration_gpt2.txt
results/calibration_qwen2.5-3b.json
results/calibration_qwen2.5-3b.txt
results/reworded_gpt2.json
results/reworded_gpt2.txt
results/reworded_qwen2.5-3b.json
results/reworded_qwen2.5-3b.txt
results/triviaqa_gpt2.jsonl
results/triviaqa_gpt2_keys.npz
results/triviaqa_gpt2_meta.json
results/triviaqa_gpt2_reworded.jsonl
results/triviaqa_gpt2_reworded_keys.npz
results/triviaqa_gpt2_reworded_meta.json
results/triviaqa_qwen2.5-3b.jsonl
results/triviaqa_qwen2.5-3b_keys.npz
results/triviaqa_qwen2.5-3b_meta.json
results/triviaqa_qwen2.5-3b_reworded.jsonl
results/triviaqa_qwen2.5-3b_reworded_keys.npz
results/triviaqa_qwen2.5-3b_reworded_meta.json
results/triviaqa_reworded.jsonl
results/triviaqa_reworded_meta.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>